In [1]:
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'datasets', 'lxml', 'cairosvg',
    'tokenizers', 'sentencepiece', 'tqdm', 'numpy'
], check=True)
print("Packages installed.")

Packages installed.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'

DIRS = [
    'data/clean',
    'data/tokenized',
    'tokenizer',
    'checkpoints',
    'logs',
    'samples',
    'configs',
]

for d in DIRS:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

print(f"\nAll directories ready under: {BASE_DIR}")


All directories ready under: /content/drive/MyDrive/svg-lm-scaling


In [4]:
import os, re, json
from lxml import etree
from datasets import load_dataset
from tqdm import tqdm

In [5]:
BASE_DIR = '/content/drive/MyDrive/svg-lm-scaling'
CLEAN_DIR = f'{BASE_DIR}/data/clean'
OUTPUT_FILE = f'{CLEAN_DIR}/all_svgs.jsonl'

MIN_CHARS = 50
MAX_CHARS = 8192   # ~1024-2048 tokens depending on tokenizer

# https://platform.openai.com/tokenizer#:~:text=translates%20to%20roughly-,%C2%BE%20of%20a%20word,-(so%20100%20tokens
# Token estimate: OpenAI reports ~4 chars/token for English (= 0.25 tokens/char).
# SVG has more varied numbers and less repetitive structure than English, so
# compression is likely worse. Using 0.25 as a conservative lower bound means
# we load more data rather than less and is safer for hitting the 100M token target.

TOKEN_ESTIMATE_RATIO = 0.25
TARGET_TOKENS = 102_000_000   # 100M training + 2M val/test

In [6]:
def clean_svg(svg_str: str):
    """Return cleaned SVG string, or None if invalid/out-of-range."""
    if not svg_str:
        return None
    svg_str = svg_str.strip()
    if len(svg_str) < MIN_CHARS or len(svg_str) > MAX_CHARS:
        return None

    try:
        parser = etree.XMLParser(remove_comments=True, remove_pis=True, recover=False)
        root = etree.fromstring(svg_str.encode('utf-8'), parser)
    except etree.XMLSyntaxError:
        return None

    # Remove noise elements
    SVG_NS = 'http://www.w3.org/2000/svg'
    for tag in ['metadata', 'title', 'desc', 'defs']:
        for elem in root.findall(f'.//{{{SVG_NS}}}{tag}'):
            parent = elem.getparent()
            if parent is not None:
                parent.remove(elem)

    etree.cleanup_namespaces(root)
    svg_out = etree.tostring(root, encoding='unicode')

    # Round floats to 1 decimal place  (reduces vocab size significantly)
    def _round(m):
        try:
            v = float(m.group())
            r = round(v, 1)
            return str(int(r)) if r == int(r) else f'{r:.1f}'
        except Exception:
            return m.group()

    svg_out = re.sub(r'-?\d+\.\d{2,}', _round, svg_out)

    # Collapse whitespace
    svg_out = re.sub(r'\s+', ' ', svg_out).strip()

    return svg_out

In [7]:
def _find_svg_field(item: dict) -> str:
    """Try common field names to find the SVG string."""
    for key in ('Svg', 'text', 'content', 'code'):
        val = item.get(key, '')
        if isinstance(val, str) and val.strip().startswith('<'):
            return val
    return ''

In [8]:
def load_and_clean(dataset_name: str, split: str = 'train',
                   max_items: int = None, streaming: bool = False) -> list:
    print(f"\nLoading  {dataset_name}  (split={split}, streaming={streaming})")
    ds = load_dataset(dataset_name, split=split, streaming=streaming)

    cleaned, skipped = [], 0
    iterator = tqdm(ds, total=max_items, desc='Cleaning')

    for item in iterator:
        if max_items and len(cleaned) + skipped >= max_items:
            break
        svg = _find_svg_field(item)
        result = clean_svg(svg)
        if result:
            cleaned.append(result)
        else:
            skipped += 1

    print(f"Kept {len(cleaned):,} Skipped {skipped:,}")
    return cleaned

In [9]:
all_svgs = []

# 1. Primary dataset (~89k icons, 153 MB)
all_svgs += load_and_clean('starvector/svg-icons-simple')

def _estimated_tokens(svgs):
    return int(sum(len(s) for s in svgs) * TOKEN_ESTIMATE_RATIO)

print(f"\nEstimated tokens so far: {_estimated_tokens(all_svgs):,}")

# 2. Emoji dataset (~14.5 MB) — load if still below target
if _estimated_tokens(all_svgs) < TARGET_TOKENS:
    all_svgs += load_and_clean('starvector/svg-emoji-simple')
    print(f"Estimated tokens so far: {_estimated_tokens(all_svgs):,}")

# 3. Stack dataset (large) — stream only what we need
if _estimated_tokens(all_svgs) < TARGET_TOKENS:
    needed_chars = (TARGET_TOKENS - _estimated_tokens(all_svgs)) / TOKEN_ESTIMATE_RATIO
    # Assume avg ~500 chars per SVG after cleaning
    max_needed = int(needed_chars / 500) + 10_000
    print(f"\nNeed ~{max_needed:,} more SVGs from svg-stack-simple (streaming)…")
    all_svgs += load_and_clean(
        'starvector/svg-stack-simple',
        max_items=max_needed,
        streaming=True,
    )
    print(f"Estimated tokens so far: {_estimated_tokens(all_svgs):,}")

print(f"\nSaving {len(all_svgs):,} SVGs to {OUTPUT_FILE}")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for svg in all_svgs:
        f.write(json.dumps({'svg': svg}) + '\n')

total_chars = sum(len(s) for s in all_svgs)
stats = {
    'total_svgs':        len(all_svgs),
    'total_chars':       total_chars,
    'estimated_tokens':  _estimated_tokens(all_svgs),
    'avg_chars_per_svg': total_chars // max(len(all_svgs), 1),
    'min_chars':         MIN_CHARS,
    'max_chars':         MAX_CHARS,
}
with open(f'{CLEAN_DIR}/stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print("\nDataset stats:")
for k, v in stats.items():
    print(f"{k}: {v:,}")


Loading  starvector/svg-icons-simple  (split=train, streaming=False)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/137M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/4.59M [00:00<?, ?B/s]

...

Cleaning: 100%|██████████| 4114/4114 [00:02<00:00, 1443.65it/s]


Kept 2,021 Skipped 2,093
Estimated tokens so far: 25,896,928

Need ~618,824 more SVGs from svg-stack-simple (streaming)…

Loading  starvector/svg-stack-simple  (split=train, streaming=True)


README.md: 0.00B [00:00, ?B/s]

Cleaning: 100%|██████████| 618824/618824 [07:25<00:00, 1389.92it/s]


Kept 518,662 Skipped 100,162
Estimated tokens so far: 189,347,411

Saving 586,214 SVGs to /content/drive/MyDrive/svg-lm-scaling/data/clean/all_svgs.jsonl

Dataset stats:
total_svgs: 586,214
total_chars: 757,389,644
estimated_tokens: 189,347,411
avg_chars_per_svg: 1,292
min_chars: 50
max_chars: 8,192


In [10]:
from datasets import load_dataset

ds = load_dataset('starvector/svg-icons-simple', split='train')
item = ds[0]

print("Field names:", list(item.keys()))
print()
for k, v in item.items():
    val_str = str(v)
    print(f"  '{k}': {val_str[:120]}")

Field names: ['Filename', 'Svg']

  'Filename': 78042
  'Svg': <svg xmlns="http://www.w3.org/2000/svg" viewBox="0.0 0.0 24.0 24.0" height="200px" width="200px"><path fill="none" strok
